# ProfileMLP (flax nnx) -- profile (F) weight recovery for DE_loopV1's ProtocolV2

Same building blocks as [MLP_Merlin.ipynb](../MLP_Merlin.ipynb)'s `SimpleMLP` (`nnx.Linear` +
`nnx.BatchNorm` + `nnx.Dropout` + `nnx.gelu`, `nnx.Optimizer` + `optax.adamw`, a warmup-cosine
LR schedule, `train_step`-style functions), wired up end-to-end to the same profile-recovery
problem as [CLAUDE/test_profile_mlp_CLAUDE.ipynb](CLAUDE/test_profile_mlp_CLAUDE.ipynb) -- i.e.
[CLAUDE/MLP_regreesionCLAUDE.py](CLAUDE/MLP_regreesionCLAUDE.py) re-implemented in JAX/flax nnx
instead of torch.

Scope (same as the torch version): profile (F) model only, single-site one-hot features -- no
pairwise (J) terms. Same toy library / `ProtocolV2` setup as
[../DE_loopV1.ipynb](../DE_loopV1.ipynb)'s "smallN" Ridge test (`N=2000`, `n_rounds=1`), so this
is directly comparable to both the Ridge and torch-MLP recoveries on the exact same protocol.

Note on `MLP_Merlin.ipynb`'s last cell: it sketches `SimpleMLP(7, 256, 140, ...)` as if the
target were the 140 profile weights themselves. A real NGS experiment never observes that --
only a scalar log-enrichment-ratio per sequence. `ProfileMLP` below outputs that scalar instead,
and F is recovered afterwards with a single-mutant scan (`extract_effective_F`).


### 0. Setup

In [ ]:
import sys, os

# This notebook lives in "Modelization_V1/MLP regression/", one level below
# sequence_classesV1.py / analysisV1.py / RegressionV1.py -- add the parent dir so their
# sibling imports resolve too (same convention as CLAUDE/test_profile_mlp_CLAUDE.ipynb).
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))

import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp

from flax import nnx
from functools import partial
from typing import Optional
import optax
from tqdm.auto import tqdm

from sequence_classesV1 import ProtocolV2, initialize_random_weights
from analysisV1 import pearson, precision_at_k, plot_teacher_vs_student
import RegressionV1 as reg

print(f"JAX backend : {jax.default_backend()} -- devices: {jax.devices()}")


## 1. Toy library

Same shape as `DE_loopV1.ipynb`'s "smallN" Ridge test (`N=2000`, `n_rounds=1`), so `ProfileMLP`,
Ridge and the torch `ProfileMLP` are all being compared on the exact same protocol.


In [ ]:
key = jax.random.key(0)
key, k_seq = jax.random.split(key)

N = 2000
sequences = jax.random.randint(k_seq, shape=(N, 7), minval=0, maxval=20)
F_viab, F_sel, J_viab, J_sel = initialize_random_weights(key)

protocol = ProtocolV2(
    N0=1_000_000, N1=500_000, dilution_factor=10, sequences=sequences, D=100_000,
    F_viab=F_viab, J_viab=J_viab, F_sel=F_sel, J_sel=J_sel,
    noise_viab=0.1, noise_sel=0.1,
)

print("protocol.sequence shape:", protocol.sequence.shape)


## 2. Single-site profile features + pooled multi-round NGS dataset

`RegressionV1.build_multi_round_dataset` already builds the pooled (sequence, log-ratio) pairs
from `protocol`'s observable NGS reads only (`lambda0p`/`lambda2p`/`lambda3p`) -- reused as-is.
Only the feature encoding changes: single-site one-hot only (`L*A = 140` dims), the profile-only
half of `RegressionV1.build_potts_features`, same as `MLP_regreesionCLAUDE.py`'s
`build_profile_features`.


In [ ]:
L = 7   # num_positions
A = 20  # num_amino_acids

def build_profile_features(seqs_obs, A=A):
    """
    Single-site one-hot features -- the (L*A,) input to ProfileMLP. Same convention as
    RegressionV1.build_potts_features' single-site block, minus the pairwise terms (this is the
    profile-only V1 scope, see MLP_regreesionCLAUDE.py's header for why).
    """
    seqs_obs = np.asarray(seqs_obs)
    N, Lseq = seqs_obs.shape
    oh = np.eye(A, dtype=np.float32)[seqs_obs]  # (N, L, A)
    return oh.reshape(N, Lseq * A)              # (N, L*A)

seqs_viab, y_viab, seqs_sel, y_sel = reg.build_multi_round_dataset(protocol, n_rounds=1)

print(f"Viability   dataset: {seqs_viab.shape[0]} (sequence, round) pairs")
print(f"Selectivity dataset: {seqs_sel.shape[0]} (sequence, round) pairs")


## 3. ProfileMLP

Same building blocks as `SimpleMLP` in `MLP_Merlin.ipynb` (Linear + BatchNorm + Dropout + gelu),
stacked twice for more capacity and extended with a `train` flag so BatchNorm/Dropout can switch
to eval mode -- `SimpleMLP`'s sketch never needed that since it was never actually run at
inference. Output is a single score (the log-ratio prediction for one sequence), not the 140
profile weights directly: those get recovered afterwards with a single-mutant scan (see
`extract_effective_F` below), the same trick used by `MLP_regreesionCLAUDE.py`'s torch version.


In [ ]:
class ProfileMLP(nnx.Module):
    """
    MLP over single-site one-hot features -> scalar score. Mirrors what Ridge's F_flat @ x does
    for the profile model, but replaces the closed-form L2 solve (which collapses toward
    near-zero weights once the feature count balloons -- see MLP_regreesionCLAUDE.py's header)
    with a network trained by SGD, same as the torch ProfileMLP but built from nnx layers.
    """

    def __init__(self, input_dim: int, hidden_dims: tuple[int, int] = (256, 128),
                 dropout_rate: float = 0.1, *, rngs: nnx.Rngs):
        h1, h2 = hidden_dims
        self.linear1    = nnx.Linear(input_dim, h1, rngs=rngs)
        self.batchnorm1 = nnx.BatchNorm(h1, use_running_average=False, rngs=rngs)
        self.dropout1   = nnx.Dropout(rate=dropout_rate, rngs=rngs)
        self.linear2    = nnx.Linear(h1, h2, rngs=rngs)
        self.batchnorm2 = nnx.BatchNorm(h2, use_running_average=False, rngs=rngs)
        self.dropout2   = nnx.Dropout(rate=dropout_rate, rngs=rngs)
        self.linear3    = nnx.Linear(h2, 1, rngs=rngs)

    def __call__(self, x: jax.Array, *, train: bool, rngs: Optional[nnx.Rngs] = None) -> jax.Array:
        x = self.linear1(x)
        x = self.batchnorm1(x, use_running_average=not train)
        x = self.dropout1(x, deterministic=not train, rngs=rngs)
        x = nnx.gelu(x)

        x = self.linear2(x)
        x = self.batchnorm2(x, use_running_average=not train)
        x = self.dropout2(x, deterministic=not train, rngs=rngs)
        x = nnx.gelu(x)

        return self.linear3(x).squeeze(-1)


### Train/eval steps + auto-adjustable LR (same recipe as MLP_Merlin.ipynb)

In [ ]:
@nnx.jit
def train_step(model, optimizer, x, y, rngs):
    """
    x : (batch, L*A) single-site one-hot features
    y : (batch,) pooled log-ratio target (viability or selectivity)
    """
    def loss_fn(model, rngs):
        y_pred = model(x, train=True, rngs=rngs)
        return jnp.mean((y_pred - y) ** 2)

    loss, grads = nnx.value_and_grad(loss_fn)(model, rngs)
    optimizer.update(model, grads)
    return loss


@nnx.jit
def eval_step(model, x, y):
    y_pred = model(x, train=False)
    return jnp.mean((y_pred - y) ** 2)


## 4. Training loop

Mini-batch SGD with the `optax.warmup_cosine_decay_schedule` from `MLP_Merlin.ipynb`, plus
early stopping on validation MSE -- `MLP_regreesionCLAUDE.py`'s torch recipe, ported to nnx:
best-val state is snapshotted with `nnx.state` and restored with `nnx.update` instead of a
torch `state_dict`.


In [ ]:
def train_profile_mlp(X_train, y_train, X_val, y_val, hidden_dims=(256, 128),
                       dropout_rate=0.1, epochs=300, batch_size=256,
                       peak_lr=1e-3, final_lr=1e-5, weight_decay=1e-4,
                       patience=20, seed=0, verbose=True):
    rngs  = nnx.Rngs(seed)
    model = ProfileMLP(input_dim=X_train.shape[1], hidden_dims=hidden_dims,
                        dropout_rate=dropout_rate, rngs=rngs)

    n_train         = X_train.shape[0]
    steps_per_epoch = max(n_train // batch_size, 1)
    total_steps     = steps_per_epoch * epochs

    lr_schedule_fn = optax.warmup_cosine_decay_schedule(
        init_value=0., peak_value=peak_lr,
        warmup_steps=int(total_steps * 0.1),
        decay_steps=int(total_steps * 0.9),
        end_value=final_lr,
    )
    optimizer = nnx.Optimizer(
        model, optax.adamw(learning_rate=lr_schedule_fn, weight_decay=weight_decay), wrt=nnx.Param
    )

    X_train, y_train = jnp.asarray(X_train), jnp.asarray(y_train)
    X_val,   y_val    = jnp.asarray(X_val),   jnp.asarray(y_val)

    shuffle_key = jax.random.key(seed)
    best_val, best_state, bad_epochs = float("inf"), None, 0
    history = {"train_loss": [], "val_loss": []}

    for epoch in tqdm(range(epochs), desc="Training ProfileMLP", disable=not verbose):
        shuffle_key, perm_key = jax.random.split(shuffle_key)
        perm = jax.random.permutation(perm_key, n_train)

        running, seen = 0.0, 0
        for i in range(steps_per_epoch):
            idx  = perm[i * batch_size : (i + 1) * batch_size]
            loss = train_step(model, optimizer, X_train[idx], y_train[idx], rngs)
            running += float(loss) * idx.shape[0]
            seen    += idx.shape[0]
        train_loss = running / seen
        val_loss   = float(eval_step(model, X_val, y_val))

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val - 1e-6:
            best_val, bad_epochs = val_loss, 0
            best_state = nnx.state(model)
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch} (best val MSE={best_val:.4f})")
                break

    nnx.update(model, best_state)
    return model, history


## 5. Train ProfileMLP for viability and selectivity

In [ ]:
def split_train_val(X, y, val_frac=0.15, seed=0):
    rng   = np.random.default_rng(seed)
    idx   = rng.permutation(len(X))
    n_val = int(len(X) * val_frac)
    val_idx, train_idx = idx[:n_val], idx[n_val:]
    return X[train_idx], y[train_idx], X[val_idx], y[val_idx]

X_viab = build_profile_features(seqs_viab)
X_sel  = build_profile_features(seqs_sel)

Xtr_v, ytr_v, Xva_v, yva_v = split_train_val(X_viab, y_viab)
Xtr_s, ytr_s, Xva_s, yva_s = split_train_val(X_sel,  y_sel)

print(f"Viability   -- train: {len(Xtr_v)}  val: {len(Xva_v)}")
print(f"Selectivity -- train: {len(Xtr_s)}  val: {len(Xva_s)}")

model_viab, hist_viab = train_profile_mlp(Xtr_v, ytr_v, Xva_v, yva_v, seed=0)
model_sel,  hist_sel  = train_profile_mlp(Xtr_s, ytr_s, Xva_s, yva_s, seed=0)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, name, hist in [(axes[0], "Viability", hist_viab),
                        (axes[1], "Selectivity", hist_sel)]:
    ax.plot(hist["train_loss"], label="train MSE")
    ax.plot(hist["val_loss"],   label="val MSE")
    ax.set_xlabel("epoch")
    ax.set_ylabel("MSE (log-ratio target)")
    ax.set_title(f"{name} -- ProfileMLP training curve")
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


## 6. Recover F via single-mutant scan

`ProfileMLP` only ever outputs one score per sequence, so the 140 profile weights are read off
by scanning every single-site mutant from an all-zero reference and taking the score delta --
same trick as `MLP_regreesionCLAUDE.py`'s `extract_effective_F`.


In [ ]:
def extract_effective_F(model, T, L=L, A=A):
    """
    Single-mutant scan from an all-zero reference sequence: predicts the score of a sequence
    with only one (position, amino acid) set to 1, offset by the all-blank prediction. Only
    recovers a meaningful F if ProfileMLP has learned an additive-across-positions function --
    exactly what training it on single-site-only features should encourage, but not a guarantee
    (same caveat as MLP_regreesionCLAUDE.py's extract_effective_F).
    """
    base    = np.zeros((1, L * A), dtype=np.float32)
    mutants = np.tile(base, (L * A, 1))
    for pos in range(L):
        for aa in range(A):
            mutants[pos * A + aa, pos * A + aa] = 1.0

    x      = jnp.asarray(np.concatenate([base, mutants], axis=0))
    scores = model(x, train=False)

    base_score = scores[0]
    F_flat     = (scores[1:] - base_score).reshape(L, A)
    return (F_flat * T).T  # (A, L), matches RegressionV1.fit_weights_potts' F_hat layout

F_viab_hat = extract_effective_F(model_viab, protocol._T_viab)
F_sel_hat  = extract_effective_F(model_sel,  protocol._T_sel)


## 7. Evaluate against profile-only ground truth

Same metrics as `RegressionV1.evaluate_recovery` / `MLP_regreesionCLAUDE.py`'s
`evaluate_profile_recovery`, but against `F_viab`/`F_sel` alone (`J` zeroed out) -- the only
thing single-site features could possibly recover.


In [ ]:
def evaluate_profile_recovery(protocol, F_viab_hat, F_sel_hat):
    v_gt  = np.array(jnp.sum(protocol.F_viab[protocol.sequence, jnp.arange(L)], axis=1))
    s_gt  = np.array(jnp.sum(protocol.F_sel[protocol.sequence,  jnp.arange(L)], axis=1))
    v_hat = np.array(jnp.sum(F_viab_hat[protocol.sequence, jnp.arange(L)], axis=1))
    s_hat = np.array(jnp.sum(F_sel_hat[protocol.sequence,  jnp.arange(L)], axis=1))

    combined_gt  = v_gt  / protocol._T_viab + s_gt  / protocol._T_sel
    combined_hat = v_hat / protocol._T_viab + s_hat / protocol._T_sel

    return dict(
        r_viab_scores      = pearson(v_gt, v_hat),
        r_sel_scores       = pearson(s_gt, s_hat),
        r_viab_weights     = pearson(np.array(protocol.F_viab).ravel(), np.array(F_viab_hat).ravel()),
        r_sel_weights      = pearson(np.array(protocol.F_sel).ravel(),  np.array(F_sel_hat).ravel()),
        precision_at_1pct  = precision_at_k(combined_gt, combined_hat, k_frac=0.01),
        precision_at_10pct = precision_at_k(combined_gt, combined_hat, k_frac=0.10),
    )

results = evaluate_profile_recovery(protocol, F_viab_hat, F_sel_hat)
for k, v in results.items():
    print(f"  {k:20s} : {v:.4f}")


In [ ]:
_ = plot_teacher_vs_student(F_viab, np.array(F_viab_hat), title="Viability: GT vs ProfileMLP-recovered")
_ = plot_teacher_vs_student(F_sel,  np.array(F_sel_hat),  title="Selectivity: GT vs ProfileMLP-recovered")
